In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json

In [21]:
load_dotenv(override=True)

True

In [22]:
openai_api_key = os.getenv('OPENAI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

In [14]:
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"

In [15]:
openai = OpenAI()
groq = OpenAI(api_key=groq_api_key,base_url = groq_url)
grok = OpenAI(api_key=grok_api_key,base_url = grok_url)

In [ ]:
tell_a_joke = [{"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"}]

In [90]:
board=[1,1,1,1,2,1,2,1,2]
X = 1
O = 2
EMPTY = 0
main_dict = {
    "cell": 'a1',
    "reasoning": "starting the game",
    "player": "X",
    "legal": True,
    "latency": 0.0,
    "tokens": 0,
}



In [ ]:
def is_legal(board, cell):
    return 0 <= cell <= 8 and board[cell] == EMPTY
is_legal(board,3)

False

In [ ]:
LINES = [
    [0,1,2],[3,4,5],[6,7,8],   # rows
    [0,3,6],[1,4,7],[2,5,8],   # cols
    [0,4,8],[2,4,6],           # diagonals
]

def winner(board):
    for a, b, c in LINES:
        if board[a] == board[b] == board[c] != EMPTY:
            return board[a]
    return None
    
print(winner(board))


1


In [63]:
def is_draw(board):
    if winner(board) is None and all(board):
        return True
    return False

is_draw(board)


0 1 2
3 4 5
6 7 8
0 3 6
1 4 7
2 5 8
0 4 8
2 4 6


True

In [43]:
def legal_moves(board):
    legal_move = []
    for i,value in enumerate(board):
        if value == EMPTY:
            legal_move.append(i)
    print(legal_move);
    return legal_move

board[3] = X;
board[4] = O;
legal_moves(board)




[5, 6, 7, 8]


[5, 6, 7, 8]

In [72]:
symbol = {0: ".", 1: "X", 2: "O"}   

def render(board):
    result = ""
    for i in range(9):
        result += symbol[board[i]] + " "   # cell ka symbol jodo
        if i == 2 or i == 5 or i == 8:     # har row ke baad nayi line
            result += "\n"
    return result

print(render(board))

X O X 
X O X 
O X O 



In [78]:
def build_prompt(board, player):
    grid = render(board)
    prompt = f"""You are playing tic-tac-toe. You are '{player}'.
The board has 9 cells numbered 0-8 (top-left is 0, bottom-right is 8).
Current board:
{grid}
Available cells: {legal_moves(board)}
Reply with ONLY JSON: {{"cell": <number>, "reasoning": "<short explanation>"}}"""
    return prompt

board = [1,2,0, 0,1,0, 0,0,0]
print(build_prompt(board, "O"))

[2, 3, 5, 6, 7, 8]
You are playing tic-tac-toe. You are 'O'.
The board has 9 cells numbered 0-8 (top-left is 0, bottom-right is 8).
Current board:
X O . 
. X . 
. . . 

Available cells: [2, 3, 5, 6, 7, 8]
Reply with ONLY JSON: {"cell": <number>, "reasoning": "<short explanation>"}


In [ ]:
def get_move(client, model, board, player):
    prompt = build_prompt(board, player)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    text = response.choices[0].message.content
    try:
        data = json.loads(text)
        return data["cell"], data["reasoning"]
    except:
        return None, text

In [95]:
def apply_move(board,cell,player):
    if is_legal(board,cell):
        board[cell] = X if player == 'X' else O
        return True
    return False
        

In [106]:
def play_game(x_client, x_model, o_client, o_model):
    board = [0]*9          # fresh board (9 zeros)
    player = "X"           # X pehle
    
    while True:
        # 1. abhi kaun khel raha? uske hisaab se client + model chuno
        if player == "X":
            client, model = x_client, x_model
        else:
            client, model = o_client, o_model
        
        # 2. move mangwao
        cell, reasoning = get_move(client, model, board, player)
        print(reasoning)

        # 3. apply karo (legal hui?)   
        ok = apply_move(board, cell, player)
        if not ok:
            print(f"{player} illegal move: {cell}")
            break
        
        # 4. board dikhao (render)
        print(render(board))

        # 5. jeet/draw check karo → jeeta to break
        w = winner(board)
        if w is not None:
            print(f"{player} Wins!")
            break
        if is_draw(board):
            print("Game Drawn")
            break
        # 6. player switch karo (X→O, O→X)
        player = "O" if player == "X" else "X"

play_game(openai, "gpt-4.1-mini", openai, "gpt-4o")

[0, 1, 2, 3, 4, 5, 6, 7, 8]
Take the center cell to maximize winning chances.
. . . 
. X . 
. . . 

Jeetni waali line: 0 1 2 → 0
Jeetni waali line: 3 4 5 → 0
Jeetni waali line: 6 7 8 → 0
Jeetni waali line: 0 3 6 → 0
Jeetni waali line: 1 4 7 → 0
Jeetni waali line: 2 5 8 → 0
Jeetni waali line: 0 4 8 → 0
Jeetni waali line: 2 4 6 → 0
Jeetni waali line: 0 1 2 → 0
Jeetni waali line: 3 4 5 → 0
Jeetni waali line: 6 7 8 → 0
Jeetni waali line: 0 3 6 → 0
Jeetni waali line: 1 4 7 → 0
Jeetni waali line: 2 5 8 → 0
Jeetni waali line: 0 4 8 → 0
Jeetni waali line: 2 4 6 → 0
[0, 1, 2, 3, 5, 6, 7, 8]
Choosing a corner cell increases chances for multiple winning paths.
O . . 
. X . 
. . . 

Jeetni waali line: 0 1 2 → 2
Jeetni waali line: 3 4 5 → 0
Jeetni waali line: 6 7 8 → 0
Jeetni waali line: 0 3 6 → 2
Jeetni waali line: 1 4 7 → 0
Jeetni waali line: 2 5 8 → 0
Jeetni waali line: 0 4 8 → 2
Jeetni waali line: 2 4 6 → 0
Jeetni waali line: 0 1 2 → 2
Jeetni waali line: 3 4 5 → 0
Jeetni waali line: 6 7 8 → 0
J